1.Installing dependencies

In [ ]:
!pip install ultralytics roboflow -q

2.Initializing Roboflow with api key

In [ ]:
from roboflow import Roboflow


car_project = rf.workspace("datasciencepro").project("self-driving-car-0mkga")
car_dataset = car_project.version(1).download("yolov8")


loading Roboflow workspace...
loading Roboflow project...


3.Checking if it initilized well

In [ ]:
import os

print(os.listdir("/content/Self-Driving-Car-1"))

['split', 'export', 'data.yaml', 'README.roboflow.txt', 'README.dataset.txt']


4.Dataset split and class remapping

In [ ]:
import os
import random
import shutil

base_path = car_dataset.location + "/export"

images_path = os.path.join(base_path, "images")
labels_path = os.path.join(base_path, "labels")

output_path = car_dataset.location + "/split"

splits = ["train", "valid", "test"]

for split in splits:
    os.makedirs(os.path.join(output_path, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(output_path, split, "labels"), exist_ok=True)

images = [f for f in os.listdir(images_path) if f.endswith(".jpg")]

random.shuffle(images)

train_split = int(0.8 * len(images))
val_split = int(0.9 * len(images))

train_files = images[:train_split]
val_files = images[train_split:val_split]
test_files = images[val_split:]

class_mapping = {
    0: 0,
    1: 1,
    2: 2,
    3: 3,
    4: 4,
    5: 4,
    6: 5,
    7: 5,
    8: 6,
    9: 6,
    10: 7
}

def process_files(files, split):

    for file in files:

        shutil.copy(
            os.path.join(images_path, file),
            os.path.join(output_path, split, "images", file)
        )

        label_file = file.replace(".jpg", ".txt")

        src_label = os.path.join(labels_path, label_file)

        dst_label = os.path.join(output_path, split, "labels", label_file)

        if os.path.exists(src_label):

            updated_lines = []

            with open(src_label, "r") as f:
                lines = f.readlines()

            for line in lines:

                parts = line.strip().split()

                if len(parts) == 0:
                    continue

                old_class = int(parts[0])

                new_class = class_mapping[old_class]

                parts[0] = str(new_class)

                updated_lines.append(" ".join(parts))

            with open(dst_label, "w") as f:
                f.write("\n".join(updated_lines))

process_files(train_files, "train")
process_files(val_files, "valid")
process_files(test_files, "test")

print("Dataset split and class remapping complete")


Dataset split and class remapping complete


5.New data.yaml creating

In [ ]:
data_yaml = """
path: /content/Self-Driving-Car-1/split

train: train/images
val: valid/images
test: test/images

nc: 8

names: [
    "biker",
    "car",
    "pedestrian",
    "trafficLight",
    "trafficLightGreen",
    "trafficLightRed",
    "trafficLightYellow",
    "truck"
]
"""

with open("data.yaml", "w") as f:
    f.write(data_yaml)

print("data.yaml created")

data.yaml created


6.Break check

In [ ]:
!cat data.yaml
!cat Self-Driving-Car-1/data.yaml


path: /content/Self-Driving-Car-1/split

train: train/images
val: valid/images
test: test/images

nc: 8

names: [
    "biker",
    "car",
    "pedestrian",
    "trafficLight",
    "green",
    "red",
    "yellow",
    "truck"
]
names:
- biker
- car
- pedestrian
- trafficLight
- trafficLight-Green
- trafficLight-GreenLeft
- trafficLight-Red
- trafficLight-RedLeft
- trafficLight-Yellow
- trafficLight-YellowLeft
- truck
nc: 11
roboflow:
  license: MIT
  project: self-driving-car-0mkga
  url: https://universe.roboflow.com/datasciencepro/self-driving-car-0mkga/dataset/1
  version: 1
  workspace: datasciencepro
test: ../test/images
train: ../train/images
val: ../valid/images


7.Break check no.2

In [ ]:
import os
from collections import Counter

labels_root = "/content/Self-Driving-Car-1/split"

class_names = {
    0: "biker",
    1: "car",
    2: "pedestrian",
    3: "trafficLight",
    4: "green",
    5: "red",
    6: "yellow",
    7: "truck"
}

counter = Counter()

for split in ["train", "valid", "test"]:

    labels_path = os.path.join(labels_root, split, "labels")

    for file in os.listdir(labels_path):

        if file.endswith(".txt"):

            with open(os.path.join(labels_path, file), "r") as f:

                lines = f.readlines()

            for line in lines:

                parts = line.strip().split()

                if len(parts) > 0:

                    class_id = int(parts[0])

                    counter[class_id] += 1

print("Class distribution:\n")

for class_id, count in sorted(counter.items()):

    print(f"{class_names[class_id]}: {count}")

Class distribution:

biker: 2489
car: 85891
pedestrian: 14284
trafficLight: 3394
green: 7697
red: 11403
yellow: 376
truck: 4822


8.Train test count and percentage

In [ ]:
import os

base_path = "/content/Self-Driving-Car-1/split"

def count_images(folder):
    return len([f for f in os.listdir(folder) if f.endswith((".jpg", ".png", ".jpeg"))])

train_count = count_images(os.path.join(base_path, "train/images"))
val_count = count_images(os.path.join(base_path, "valid/images"))
test_count = count_images(os.path.join(base_path, "test/images"))

total = train_count + val_count + test_count

print(f"Train: {train_count}")
print(f"Valid: {val_count}")
print(f"Test:  {test_count}")
print(f"Total: {total}")

print("\nPercentages:")
print(f"Train: {train_count/total:.2%}")
print(f"Valid: {val_count/total:.2%}")
print(f"Test:  {test_count/total:.2%}")

Train: 12745
Valid: 2510
Test:  2517
Total: 17772

Percentages:
Train: 71.71%
Valid: 14.12%
Test:  14.16%


9.Training

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="/content/data.yaml",
    epochs=100,
    patience=15,
    imgsz=640,
    batch=16,
    name="car_model",

    fliplr=0.5,

    hsv_v=0.1,
    hsv_s=0.1,

    mosaic=1.0,
    close_mosaic=10,

    mixup=0.0,

    degrees=0.0
)

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.1, hsv_v=0.1, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=car_model, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=15, perspe

KeyboardInterrupt: 

10.Check if it saved best.pt

In [ ]:
import os

base_path = "/content/runs/detect/car_model/weights"
print(os.listdir(base_path))

['best.pt', 'last.pt']


11.Priniting metrics

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/car_model/weights/best.pt")
metrics = model.val(data="/content/data.yaml", imgsz=640, device=0)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)


# model = YOLO("/content/best (1).pt")

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,128,680 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1427.7±494.8 MB/s, size: 46.8 KB)
val: Scanning /content/Self-Driving-Car-1/split/valid/labels.cache... 2510 images, 8 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2510/2510 658.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 157/157 5.6it/s 28.0s
                   all       2510      18447      0.849       0.72      0.784      0.472
                 biker        224        354      0.828      0.701      0.751      0.456
                   car       2453      12141        0.9      0.811      0.857      0.589
            pedestrian        662       2064      0.817      0.594      0.682       0.36
          trafficLight        283        451      0.869      0.795      0.855      0.557
                 green 

12.Downloading best.pt

In [ ]:
!ls runs/detect/
from google.colab import files

files.download("runs/detect/car_model/weights/best.pt")
# files.download('runs/classify/car_model/weights/best.pt')

ls: cannot access 'runs/detect/': No such file or directory


FileNotFoundError: Cannot find file: runs/detect/car_model/weights/best.pt